In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OrdinalEncoder
import seaborn as sns
from scipy.stats import gaussian_kde



In [ ]:
ames = pd.read_csv("../Data/AmesHousing.csv")
ames = ames.drop(columns=['Pool QC', 'Fence', 'Misc Feature', 'Alley'])

In [ ]:
ames.isnull().sum()

In [ ]:
ames.columns

## Visualizing seaborn(and matplotlib) 

In [ ]:
sns.set_theme(style='whitegrid')
plt.figure(figsize=(10, 6))

# cmap='viridis' is colorblind-friendly. cbar=False removes the legend.
sns.heatmap(ames.isnull(), cbar=False, cmap='viridis', yticklabels=False)

plt.title('Missing Data Map (Yellow = Missing)', fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

# 1. Filter to keep ONLY columns that have at least one NaN
missing_cols = ames.columns[ames.isnull().any()]
df_missing_only = ames[missing_cols]

# 2. Plot the heatmap using the filtered DataFrame
sns.heatmap(
    df_missing_only.isnull(), 
    cbar=False, 
    cmap='viridis', 
    yticklabels=False
)

plt.title('Missing Data Map (Filtered)', fontsize=14)
plt.show()

In [ ]:
target = 'SalePrice'
skewness = ames[target].skew()

plt.figure(figsize=(10, 6))
sns.histplot(ames[target], kde=True, color='blue', bins=40)

# Display skewness directly in the title
plt.title(f'Distribution of {target}\nSkewness: {skewness:.3f}', fontsize=14)
plt.xlabel(f'{target} ($)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Number of variables for heatmap
k = 5 

# 1. Filter out categorical columns to avoid warnings/errors
numeric_df = ames.select_dtypes(include=[np.number])

# 2. Find the top 'k' correlated features with the target
corrmat = numeric_df.corr()
top_cols = corrmat.nlargest(k, target)[target].index

# 3. Calculate correlation matrix for just those top features
cm = np.corrcoef(numeric_df[top_cols].values.T)

# 4. Plot the heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True,              # Show the actual correlation numbers
    fmt='.2f',               # Format to 2 decimal places
    cmap='coolwarm',         # Blue (negative) to Red (positive)
    yticklabels=top_cols, 
    xticklabels=top_cols,
    square=True,             # Keep the cells square
    linewidths=0.5
)

plt.title(f'Top {k} Predictors Correlation Matrix', fontsize=14)
plt.xticks(rotation=45, ha='right') # Rotate x labels for better readability
plt.show()

## Visualizing with matplotlib only. 

In [ ]:
plt.figure(figsize=(10, 6))

# Use imshow to plot the boolean mask
plt.imshow(ames.isnull(), aspect='auto', cmap='viridis', interpolation='none')

# Clean up the axes to match the seaborn look
plt.yticks([]) # Remove y-axis ticks (row numbers)
plt.xticks(ticks=np.arange(len(ames.columns)), labels=ames.columns, rotation=45, ha='right')

plt.title('Missing Data Map (Yellow = Missing)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

# 1. Filter to keep ONLY columns that have at least one NaN
missing_cols = ames.columns[ames.isnull().any()]
df_missing_only = ames[missing_cols]

# 2. Plot the boolean mask of the filtered DataFrame
plt.imshow(df_missing_only.isnull(), aspect='auto', cmap='viridis', interpolation='none')

# 3. Clean up the axes
plt.yticks([]) # Remove y-axis ticks (row numbers)

# Set x-ticks to match our newly filtered column list
plt.xticks(ticks=np.arange(len(missing_cols)), labels=missing_cols, rotation=45, ha='right')

plt.title('Missing Data Map (Filtered)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
target = 'SalePrice'
skewness = ames[target].skew()

plt.figure(figsize=(10, 6))

# 1. Plot the histogram (density=True normalizes it so the KDE curve scales correctly)
plt.hist(ames[target], bins=40, density=True, color='cornflowerblue', alpha=0.7, edgecolor='white')

# 2. Calculate and plot the KDE curve
kde = gaussian_kde(ames[target])
x_vals = np.linspace(ames[target].min(), ames[target].max(), 1000)
plt.plot(x_vals, kde(x_vals), color='darkblue', linewidth=2)

plt.title(f'Distribution of {target}\nSkewness: {skewness:.3f}', fontsize=14)
plt.xlabel(f'{target} ($)')
plt.ylabel('Density')
plt.show()

In [ ]:
k = 5 
numeric_df = ames.select_dtypes(include=[np.number])
corrmat = numeric_df.corr()
top_cols = corrmat.nlargest(k, target)[target].index
cm = np.corrcoef(numeric_df[top_cols].values.T)

fig, ax = plt.subplots(figsize=(8, 6))

# 1. Plot the color grid
cax = ax.imshow(cm, cmap='coolwarm', vmin=-1, vmax=1)

# 2. Add the colorbar on the side
fig.colorbar(cax)

# 3. Set up the X and Y axes labels
ax.set_xticks(np.arange(len(top_cols)))
ax.set_yticks(np.arange(len(top_cols)))
ax.set_xticklabels(top_cols, rotation=45, ha='right')
ax.set_yticklabels(top_cols)

# 4. Loop over data dimensions and create text annotations
for i in range(len(top_cols)):
    for j in range(len(top_cols)):
        # Change text color to white if the square is too dark for black text
        text_color = "white" if abs(cm[i, j]) > 0.6 else "black"
        ax.text(j, i, f"{cm[i, j]:.2f}", 
                ha="center", va="center", color=text_color)

plt.title(f'Top {k} Predictors Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## ...